In [ ]:
# Inundation Area Identification using Elevation (Shapefile) and Shoreline (Shapefile)


# ============================================================================
# CELL 1: Install Required Libraries
# ============================================================================
"""
!pip install geopandas
!pip install shapely
"""

# ============================================================================
# CELL 2: Import Libraries
# ============================================================================
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CELL 3: Mount Google Drive
# ============================================================================
from google.colab import drive

drive.mount('/content/drive')
print("Google Drive mounted successfully!")

# ============================================================================
# CELL 4: Define Inundation Ranking Function
# ============================================================================
def get_inundation_rank(elevation, distance_band):
    """
    Determine inundation rank based on elevation and distance band from shoreline.

    Ranking Table:
    Distance(m) | 0-1m  | 1-2m    | >2m
    0-500       | High  | Medium  | Low
    500-1000    | Medium| Low     | Low
    1000-1500   | Low   | Low     | Low
    >1500       | No_Risk (outside 1.5km cutoff; adjust if you want a 1500-2000 band)
    """
    if pd.isna(elevation):
        return 'NoData'

    if distance_band == '0-500m':
        if elevation <= 1:
            return 'High'
        elif elevation <= 2:
            return 'Medium'
        else:
            return 'Low'

    elif distance_band == '500-1000m':
        if elevation <= 1:
            return 'Medium'
        else:
            return 'Low'

    elif distance_band in ('1000-1500m', '1500-2000m'):
        return 'Low'

    else:
        return 'No_Risk'

# ============================================================================
# CELL 5: Load Data from Google Drive
# ============================================================================
# MODIFY THESE PATHS TO YOUR GOOGLE DRIVE PATHS
elevation_shp_file = '/content/drive/My Drive/AP_Inputs/Elevation/merged_shapefile.shp'
elevation_attribute = 'Contour'  # Change to your elevation column name
shoreline_file = '/content/drive/My Drive/AP_Inputs/Shoreline-2025.shp'

output_dir = '/content/drive/My Drive/AP_Inputs/outputs'

import os
os.makedirs(output_dir, exist_ok=True)

print("Loading elevation shapefile...")
elevation_gdf = gpd.read_file(elevation_shp_file)
print(f"  CRS: {elevation_gdf.crs}")
print(f"  Number of features: {len(elevation_gdf)}")
print(f"  Columns: {list(elevation_gdf.columns)}")
print(f"  Geometry type(s): {elevation_gdf.geometry.type.unique()}")

if elevation_attribute in elevation_gdf.columns:
    print(f"\n{elevation_attribute} statistics:")
    print(f"  Min: {elevation_gdf[elevation_attribute].min()}")
    print(f"  Max: {elevation_gdf[elevation_attribute].max()}")
    print(f"  Mean: {elevation_gdf[elevation_attribute].mean():.2f}")
else:
    print(f"\nWarning: Column '{elevation_attribute}' not found!")
    print(f"Available columns: {list(elevation_gdf.columns)}")
    elevation_attribute = input("Enter the correct elevation column name: ")

print(f"\nLoading shoreline shapefile...")
shoreline_gdf = gpd.read_file(shoreline_file)
print(f"  CRS: {shoreline_gdf.crs}")
print(f"  Number of features: {len(shoreline_gdf)}")

# Reproject shoreline to match elevation CRS if needed
if elevation_gdf.crs != shoreline_gdf.crs:
    print("\nReprojecting shoreline to match elevation CRS...")
    shoreline_gdf = shoreline_gdf.to_crs(elevation_gdf.crs)
    print("Reprojection complete")

# IMPORTANT: distances below are in the units of elevation_gdf.crs.
# If your CRS is a projected metric CRS (e.g. UTM), units are meters and
# the buffer distances (500, 1000, etc.) are correct as-is.
# If your CRS is geographic (lat/lon, e.g. EPSG:4326), buffering in degrees
# is WRONG. Reproject both layers to a local UTM zone first, e.g.:
#   elevation_gdf = elevation_gdf.to_crs(epsg=32644)  # adjust EPSG to your UTM zone
#   shoreline_gdf = shoreline_gdf.to_crs(epsg=32644)
print(f"\nWorking CRS units check - is_geographic: {elevation_gdf.crs.is_geographic}")
if elevation_gdf.crs.is_geographic:
    print("  WARNING: Your CRS is geographic (degrees). Buffer distances in")
    print("  meters will be WRONG. Reproject to a metric/UTM CRS before continuing.")

# ============================================================================
# CELL 6: Fix Invalid Geometries and Simplify (common cause of crashes in overlay/buffer)
# ============================================================================
print("\nValidating, repairing, and simplifying geometries...")

# Simplify geometries to reduce vertex count and memory usage for large datasets
# Adjust `tolerance` based on desired precision. E.g., 1 meter for general analysis.
print("  Simplifying elevation geometries...")
elevation_gdf['geometry'] = elevation_gdf.geometry.simplify(1, preserve_topology=True) # Tolerance in CRS units (e.g., meters)

elevation_gdf['geometry'] = elevation_gdf.geometry.buffer(0)
elevation_gdf = elevation_gdf[elevation_gdf.geometry.notna() & ~elevation_gdf.geometry.is_empty]
elevation_gdf = elevation_gdf.reset_index(drop=True)

shoreline_gdf = shoreline_gdf[shoreline_gdf.geometry.notna() & ~shoreline_gdf.geometry.is_empty]
shoreline_gdf = shoreline_gdf.reset_index(drop=True)

print(f"  Elevation features after cleanup: {len(elevation_gdf)}")
print(f"  Shoreline features after cleanup: {len(shoreline_gdf)}")

# ============================================================================
# CELL 7: Build Distance Bands from Shoreline (Vector Buffers, No Raster)
# ============================================================================
print("\nBuilding distance bands from shoreline (vector buffer rings)...")

# Dissolve all shoreline segments into a single geometry for buffering
shoreline_union = unary_union(shoreline_gdf.geometry)

band_distances = [500, 1000, 1500, 2000]
band_labels = ['0-500m', '500-1000m', '1000-1500m', '1500-2000m']

buffers = [shoreline_union.buffer(d) for d in band_distances]

ring_records = []
previous_buffer = None
for buf, label in zip(buffers, band_labels):
    ring_geom = buf if previous_buffer is None else buf.difference(previous_buffer)
    ring_records.append({'distance_band': label, 'geometry': ring_geom})
    previous_buffer = buf

bands_gdf = gpd.GeoDataFrame(ring_records, crs=elevation_gdf.crs)
bands_gdf['area_m2'] = bands_gdf.geometry.area
bands_gdf['area_km2'] = bands_gdf['area_m2'] / 1_000_000

print("Distance band areas:")
for _, row in bands_gdf.iterrows():
    print(f"  {row['distance_band']}: {row['area_km2']:.2f} km²")

# Save distance bands as their own shapefile
bands_shp = os.path.join(output_dir, 'distance_bands.shp')
bands_gdf.to_file(bands_shp)
print(f"Distance bands shapefile saved: {bands_shp}")

# ============================================================================
# CELL 8: Overlay Elevation Polygons with Distance Bands
# ============================================================================
print("\nOverlaying elevation polygons with distance bands...")

# Intersection splits elevation polygons wherever a distance-band boundary
# crosses them, so each output polygon has both an elevation value and a
# single distance band - no rasterization needed.
overlay_gdf = gpd.overlay(
    elevation_gdf[[elevation_attribute, 'geometry']],
    bands_gdf[['distance_band', 'geometry']],
    how='intersection',
    keep_geom_type=True
)

print(f"  Polygons after overlay: {len(overlay_gdf)}")

# ============================================================================
# CELL 9: Apply Inundation Ranking
# ============================================================================
print("\nApplying inundation ranking...")

overlay_gdf['risk_level'] = overlay_gdf.apply(
    lambda row: get_inundation_rank(row[elevation_attribute], row['distance_band']),
    axis=1
)

rank_to_numeric = {'High': 3, 'Medium': 2, 'Low': 1, 'No_Risk': 0, 'NoData': -9999}
overlay_gdf['risk_code'] = overlay_gdf['risk_level'].map(rank_to_numeric)

print("\nUnique classes:")
print(overlay_gdf['risk_level'].value_counts())

# ============================================================================
# CELL 10: Calculate Area Per Polygon and Dissolve by Risk Level
# ============================================================================
overlay_gdf['area_m2'] = overlay_gdf.geometry.area
overlay_gdf['area_km2'] = overlay_gdf['area_m2'] / 1_000_000
overlay_gdf = overlay_gdf.rename(columns={elevation_attribute: 'elevation_m'})

# Full detailed output: every split polygon keeps its elevation + band + risk
detailed_shp = os.path.join(output_dir, 'inundation_risk_areas_detailed.shp')
overlay_gdf.to_file(detailed_shp)
print(f"Detailed inundation shapefile saved: {detailed_shp}")

dissolved_shp = None # Initialize to None in case it's skipped
if not _SKIP_DISSOLVE_OPERATION:
    print("\nDissolving by risk level...")
    # Dissolved output: one polygon per risk level (cleaner for mapping/summary)
    dissolved_gdf = overlay_gdf.dissolve(by='risk_level', aggfunc={
        'area_m2': 'sum',
        'area_km2': 'sum'
    }).reset_index()
    dissolved_gdf['risk_code'] = dissolved_gdf['risk_level'].map(rank_to_numeric)

    dissolved_shp = os.path.join(output_dir, 'inundation_risk_areas_dissolved.shp')
    dissolved_gdf.to_file(dissolved_shp)
    print(f"Dissolved inundation shapefile saved: {dissolved_shp}")
else:
    print("\nSkipping dissolved shapefile generation (as _SKIP_DISSOLVE_OPERATION is True).")


# ============================================================================
# CELL 11: Calculate Statistics
# ============================================================================
print("\n" + "=" * 60)
print("INUNDATION AREA STATISTICS")
print("=" * 60)

total_area_km2 = overlay_gdf['area_km2'].sum()

for rank in ['High', 'Medium', 'Low']:
    subset = overlay_gdf[overlay_gdf['risk_level'] == rank]
    area_km2 = subset['area_km2'].sum()
    percent = (area_km2 / total_area_km2 * 100) if total_area_km2 > 0 else 0
    print(f"\n{rank} Risk Zone:")
    print(f"  Polygons: {len(subset)}")
    print(f"  Area: {area_km2:.2f} km²")
    print(f"  Percentage: {percent:.2f}%")

print(f"\nTotal Analysis Area (within 2km of shoreline): {total_area_km2:.2f} km²")

# ============================================================================
# CELL 12: Export High Risk Areas ONLY as Separate Shapefile
# ============================================================================
print("\n" + "=" * 60)
print("EXTRACTING HIGH RISK AREAS")
print("=" * 60)

high_risk_gdf = overlay_gdf[overlay_gdf['risk_level'] == 'High'].copy()

if len(high_risk_gdf) > 0:
    high_risk_shp = os.path.join(output_dir, 'high_risk_inundation_areas.shp')
    high_risk_gdf.to_file(high_risk_shp)
    print(f"High risk shapefile saved: {high_risk_shp}")
    print(f"  Number of high-risk polygons: {len(high_risk_gdf)}")
    print(f"  Total high-risk area: {high_risk_gdf['area_km2'].sum():.2f} km²")
else:
    print("No high-risk areas found in the analysis.")

# ============================================================================
# CELL 13: Visualize Results
# ============================================================================
print("\nGenerating visualization...")

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

risk_colors = {'High': 'red', 'Medium': 'orange', 'Low': 'yellow', 'No_Risk': 'lightgray'}

for risk, color in risk_colors.items():
    subset = overlay_gdf[overlay_gdf['risk_level'] == risk]
    if len(subset) > 0:
        subset.plot(ax=ax, color=color, edgecolor='none', label=risk)

shoreline_gdf.plot(ax=ax, color='blue', linewidth=1.5, label='Shoreline')

ax.set_title('Inundation Risk Classification', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.set_aspect('equal')

png_path = os.path.join(output_dir, 'inundation_analysis_overview.png')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"Map saved: {png_path}")

# ============================================================================
# CELL 14: Summary Report
# ============================================================================
print("\n" + "=" * 70)
print("ANALYSIS COMPLETE - SUMMARY OF OUTPUTS")
print("=" * 70)
print(f"\nOutput Directory: {output_dir}")
print(f"\nGenerated Files:")
print(f"  - Distance Bands (Shapefile): {bands_shp}")
print(f"  - Detailed Inundation Risk Areas (Shapefile): {detailed_shp}")
if dissolved_shp: # Only print if it was generated
    print(f"  - Dissolved Inundation Risk Areas (Shapefile): {dissolved_shp}")
if len(high_risk_gdf) > 0:
    print(f"  - High Risk Inundation Areas (Shapefile): {high_risk_shp}")
print(f"  - Inundation Risk Map (PNG): {png_path}")

In [1]:
# Set to True to skip the memory-intensive dissolve operation in CELL 10
# Set to False if you have enough RAM and wish to generate the dissolved output.
_SKIP_DISSOLVE_OPERATION = True